In [1]:
from flask import Flask, request, jsonify
import mysql.connector
import threading
import uuid

def get_db():
    return mysql.connector.connect(
        host="localhost", 
        user="root", 
        password="", 
        database="gadgethub_db"
    )

def get_quote_from_db(distributor_name, product_ids):
    
    conn = get_db()
    cursor = conn.cursor(dictionary=True)
    format_strings = ','.join(['%s'] * len(product_ids))
    query = f"""
        SELECT i.product_id, i.price, i.stock, p.name 
        FROM distributor_inventory i
        JOIN products p ON i.product_id = p.id
        WHERE i.distributor_name = %s AND i.product_id IN ({format_strings}) AND i.stock > 0
    """
    params = [distributor_name] + product_ids
    cursor.execute(query, params)
    results = cursor.fetchall()
    conn.close()
    
    response = {}
    for row in results:
        response[row['product_id']] = {
            "distributor": distributor_name,
            "product_name": row['name'],
            "price": row['price'],
            "stock": row['stock'],
            "eta_days": row.get('eta_days', 3)
        }
    return response
    
   # Reusable distributor order processing
def process_distributor_order(distributor_name, data):
    # Used by all three distributor services
    # Single implementation, multiple consumers
    
    # Requirement: "Reserve/reduce stock" and "Return distributor-specific order IDs"
    product_id = data.get('product_id')
    quantity = data.get('quantity', 1)
    
    conn = get_db()
    cursor = conn.cursor()
    
    # Check Stock
    cursor.execute("SELECT stock FROM distributor_inventory WHERE distributor_name=%s AND product_id=%s", 
    (distributor_name, product_id))
    row = cursor.fetchone()
    
    if not row or row[0] < quantity:
        conn.close()
        return {"status": "Failed", "error": "Out of Stock"}, 400
        
    # Reduce Stock
    cursor.execute("UPDATE distributor_inventory SET stock = stock - %s WHERE distributor_name=%s AND product_id=%s", 
    (quantity, distributor_name, product_id))
    conn.commit()
    conn.close()
    
    # Generate Distributor Order ID (e.g., "TW-1234")
    prefix = distributor_name[:2].upper()
    dist_order_id = f"{prefix}-{str(uuid.uuid4())[:8]}"
    
    return {"status": "Confirmed", "distributor_order_id": dist_order_id}, 200

# --- APP SETUP ---

app_tech = Flask("TechWorld")
@app_tech.route('/quote', methods=['POST'])
def quote_tech(): return jsonify(get_quote_from_db("TechWorld", request.json['product_ids']))

@app_tech.route('/order', methods=['POST']) # <--- NEW ENDPOINT
def order_tech(): 
    resp, code = process_distributor_order("TechWorld", request.json)
    return jsonify(resp), code

app_electro = Flask("ElectroCom")
@app_electro.route('/quote', methods=['POST'])
def quote_electro(): return jsonify(get_quote_from_db("ElectroCom", request.json['product_ids']))

@app_electro.route('/order', methods=['POST']) # <--- NEW ENDPOINT
def order_electro():
    resp, code = process_distributor_order("ElectroCom", request.json)
    return jsonify(resp), code

app_gadget = Flask("GadgetCentral")
@app_gadget.route('/quote', methods=['POST'])
def quote_gadget(): return jsonify(get_quote_from_db("GadgetCentral", request.json['product_ids']))

@app_gadget.route('/order', methods=['POST']) # <--- NEW ENDPOINT
def order_gadget():
    resp, code = process_distributor_order("GadgetCentral", request.json)
    return jsonify(resp), code

# Run threads
threading.Thread(target=lambda: app_tech.run(port=5001, use_reloader=False)).start()
threading.Thread(target=lambda: app_electro.run(port=5002, use_reloader=False)).start()
threading.Thread(target=lambda: app_gadget.run(port=5003, use_reloader=False)).start()
print("✅ Distributors Running (5001, 5002, 5003) with Order Processing...")

✅ Distributors Running (5001, 5002, 5003) with Order Processing...
 * Serving Flask app 'TechWorld'
 * Debug mode: off
 * Serving Flask app 'ElectroCom'
 * Debug mode: off
 * Serving Flask app 'GadgetCentral'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [16/Feb/2026 14:13:03] "POST /quote HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2026 14:13:03] "POST /quote HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2026 14:13:03] "POST /quote HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2026 14:13:03] "POST /order HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2026 14:13:03] "POST /order HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2026 14:13:08] "GET /quote HTTP/1.1" 405 -
127.0.0.1 - - [16/Feb/2026 14:13:22] "GET /quote HTTP/1.1" 405 -
